# Stage 1 — wallet strategy selection (unified sports)

One full uncarved BUY stream drives everything:

- per-wallet stats are computed on the **train** BUY fills (`last_condition_trade_ts <= TRAIN_END`)
- wallet cohorts come from `signal_lab.filters` (`select_strategy_eligible` / `select_strategy_selection`)
- every strategy is a **mask** on the same BUY stream (optional simple transformation)
- there is no exposure carve and no `/` ``*_limited`` PnL bookkeeping
- output: `selected_wallets_v2.parquet` with `copyable_group` / `predicting_group` for stage2


## Configuration

All knobs live here. Strategy overlays are fitted on the notebook's train split only.


In [1]:
%load_ext autoreload
%autoreload 2

import sys
from enum import Enum
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import pyarrow.dataset as ds

np.set_printoptions(precision=4, suppress=True)
pd.options.display.float_format = "{:.4f}".format


class MarketFilterMode(Enum):
    DEFAULT = "default"
    SPORTS = "sports"
    CRYPTO = "crypto"


MARKET_FILTER_MODE = MarketFilterMode.SPORTS

# TAGS = {"Weather"}
# TAGS = {"Esports"}
TAGS = {"Sports"}
# TAGS = {"Politics"}

MAX_LEAD_DAYS = 14 if "Politics" in TAGS else 7

if MARKET_FILTER_MODE == MarketFilterMode.SPORTS:
    TRADES_DIR = Path("../../data/polygon_trades_processed_sports")
    WORKSPACE_DIR = Path("../../data/trade_signals_workspace_v2_sports")
elif MARKET_FILTER_MODE == MarketFilterMode.CRYPTO:
    TRADES_DIR = Path("../../data/polygon_trades_processed_crypto")
    WORKSPACE_DIR = Path("../../data/trade_signals_workspace_v2_crypto")
else:
    TRADES_DIR = Path("../../data/polygon_trades_processed")
    WORKSPACE_DIR = Path("../../data/trade_signals_workspace_v2")
WORKSPACE_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_END = pd.Timestamp("2026-09-01", tz="UTC")

# Basic train-window wallet screen (applied before computing wallet stats).
MIN_TRAIN_PNL = 1_000
MIN_TRAIN_MARKETS = 20

# Contrarian flow overlay.
FLOW_PRICE_LO = 0.20
FLOW_PRICE_HI = 0.50
FLOW_SHARE_THR = -0.10
FLOW_STRICT = False

# Size-share overlay.
SIZE_PRICE_LO = 0.30
SIZE_PRICE_HI = 0.75
SIZE_SHARE_QUANTILE = 0.50

# Flow-selection port (exact stage1_flow_selection constants).
FLOWSEL_CANDIDATE_MAX_LEAD_DAYS = 2
FLOWSEL_MIN_LEAD_DAYS = 0
FLOWSEL_MAX_LEAD_DAYS = 7
FLOWSEL_FIT_END = pd.Timestamp("2026-08-01", tz="UTC")
FLOWSEL_VAL_END = pd.Timestamp("2026-09-01", tz="UTC")
FLOWSEL_PRICE_BAND = (0.30, 0.75)
FLOWSEL_SIZE_SHARE_QUANTILE = 0.50
FLOWSEL_MIN_FIT_WALLET_FILLS = 20
FLOWSEL_MIN_VAL_WALLET_FILLS = 10


## Load markets and BUY fills

Reads only `side == "BUY"` rows for markets that match `TAGS`, keeps fills
`0 <= lead <= MAX_LEAD_DAYS`, derives canonical columns, and applies the basic
train-window wallet screen.


In [2]:
from polymarket_analysis.data.data_catalogue import load_markets_processed

mdf = load_markets_processed()
_tagset = set(TAGS)
markets = mdf[
    mdf["tags"].apply(lambda tags: bool(_tagset & set(tags)))
    & ~mdf["primary_tag"].isin(["Crypto"])
    & mdf["winner_token_id"].notna()
].copy()
conditions = list(set(markets["condition_id"]))
print(f"markets for tags {TAGS}: {len(conditions):,}")


markets for tags {'Sports'}: 3,265,838


In [3]:
FILL_COLUMNS = [
    "wallet", "condition_id", "token_id", "tx_hash", "dt", "side",
    "avg_price", "trade_value_usdc", "total_quantity", "position",
    "copyable_pnl", "copyable_qty_5m_100", "avail_copy_total_vol_5m_100",
    "flow_own_usdc_60", "usdc_all_60", "price_chg_last_own",
    "last_condition_trade_ts", "final_value_usdc",
]

dataset = ds.dataset(TRADES_DIR, format="parquet")
df = (
    dataset.to_table(
        columns=FILL_COLUMNS,
        filter=(ds.field("side") == "BUY") & ds.field("condition_id").isin(conditions),
    )
    .to_pandas()
)
df["wallet"] = df["wallet"].astype(str)
df = df.rename(
    columns={
        "avg_price": "price",
        "trade_value_usdc": "usdc_amount",
        "total_quantity": "quantity",
    }
)
df["dt"] = pd.to_datetime(df["dt"], utc=True)
df["last_condition_trade_ts"] = pd.to_datetime(df["last_condition_trade_ts"], utc=True)

lead = df["last_condition_trade_ts"] - df["dt"]
df = df[(lead >= pd.Timedelta(0)) & (lead <= pd.Timedelta(days=MAX_LEAD_DAYS))].copy()

_float_cols = [
    "price", "usdc_amount", "quantity", "position", "copyable_pnl",
    "copyable_qty_5m_100", "avail_copy_total_vol_5m_100", "final_value_usdc",
]
df[_float_cols] = df[_float_cols].astype(float)

df["pnl"] = df["final_value_usdc"] - df["usdc_amount"]
df["notional"] = df["usdc_amount"]
df["copyable_notional"] = df["price"] * df["copyable_qty_5m_100"].clip(lower=0)
df["exposure"] = np.where(
    df["copyable_qty_5m_100"] > 0,
    df["price"] * df["copyable_qty_5m_100"],
    np.nan,
)
df["flow_usdc_share_60"] = (
    df["flow_own_usdc_60"] / df["usdc_all_60"].replace(0, np.nan)
)
df["size_share"] = (
    df["usdc_amount"].astype(float) / df["usdc_all_60"].astype(float).replace(0, np.nan)
)
df["is_train"] = df["last_condition_trade_ts"] <= TRAIN_END

_n_wallets = df["wallet"].nunique()
_train = df[df["is_train"]]
_train_stats = _train.groupby("wallet").agg(
    train_pnl=("pnl", "sum"),
    train_markets=("condition_id", "nunique"),
)
del _train
_keep = _train_stats.index[
    (_train_stats["train_pnl"] >= MIN_TRAIN_PNL)
    & (_train_stats["train_markets"] >= MIN_TRAIN_MARKETS)
]
df = df[df["wallet"].isin(_keep)].reset_index(drop=True)

print(f"BUY fills: {len(df):,}")
print(f"wallets: {df['wallet'].nunique():,} of {_n_wallets:,}")
print(f"train/test fills: {int(df['is_train'].sum()):,} / {int((~df['is_train']).sum()):,}")


BUY fills: 16,128,444


wallets: 6,704 of 21,250
train/test fills: 13,292,621 / 2,835,823


## Wallet stats (train BUY fills)

A single stat frame feeds both the strategy masks and the explorer. `buy_copyable_roi`
is derived here because `compute_wallet_metrics` does not emit it.


In [4]:
from polymarket_analysis.wallet_selection.volatility import compute_wallet_metrics

wallet_stats, _ = compute_wallet_metrics(df.loc[df["is_train"]].copy())
wallet_stats["buy_copyable_roi"] = (
    wallet_stats["buy_copyable_pnl"]
    / wallet_stats["buy_copyable_notional"].replace(0, np.nan)
)
wallet_stats = wallet_stats.sort_values("buy_copyable_roi", ascending=False).reset_index(drop=True)
print(f"wallet stats rows: {len(wallet_stats):,}")
wallet_stats.head()


wallet stats rows: 6,704


,wallet,num_buckets,trade_count,total_notional,total_pnl,copyable_pnl,num_markets,median_dt,median_roi,average_roi,...,copyable_pnl_similarity,copyable_pnl_similarity_profit,copyable_pnl_similarity_loss,copyable_pnl_similarity_20m_100,copyable_pnl_similarity_profit_20m_100,copyable_pnl_similarity_loss_20m_100,max_drawdown_to_pnl,max_copyable_drawdown_to_copyable_pnl,return,buy_copyable_roi
0,0x817ad2a74005166cefc04e0ae2ebd5b8b18b7e90,34,34,4119.8903,8290.9784,8619.9108,30,2026-04-28 03:00:00+00:00,-1.0000,0.8698,...,0.9921,0.9980,0.5618,NaN,NaN,NaN,0.0772,0.0429,2.0124,18.6842
1,0xc258160e3bbddbc6cdce69cca3b722e7df59bbe6,44,80,805.7586,5562.6548,1241.2036,23,2026-05-08 12:47:30+00:00,-1.0000,0.5284,...,0.6807,0.6812,0.6188,NaN,NaN,NaN,0.0479,0.0721,6.9036,7.0783
2,0xdd4056f53e031c2663e7b98806cb1fdc560812d0,40,56,1233.8186,2381.0958,2584.5325,26,2026-01-25 09:00:00+00:00,-1.0000,0.2453,...,0.9982,1.0000,0.9211,NaN,NaN,NaN,0.1893,0.1228,1.9299,4.8215
3,0xc3ae65d03ff61aa3676a99bb045669a823f733b8,45,45,2950.3134,1620.5458,1981.0697,42,2026-07-06 22:00:00+00:00,-1.0000,0.6331,...,0.9382,0.9911,0.6454,NaN,NaN,NaN,0.4613,0.1668,0.5493,4.8007
4,0x5ae3d9d04cd44699e6316ab8053d2d05c007d88a,321,721,12902.9378,19974.8776,7909.5009,299,2026-07-26 12:35:00+00:00,-1.0000,12.0116,...,0.8703,0.8720,0.4826,NaN,NaN,NaN,0.0355,0.0600,1.5481,4.7313


## Wallet selection

Both cohorts are disjoint subsets of the base-eligible wallets. Selection is
train-only and uncarved.


In [5]:
sys.path.insert(0, str(Path.cwd().parent / "wallet_selection"))

from signal_lab.filters import select_strategy_eligible, select_strategy_selection

if "Esports" in TAGS or "Sports" in TAGS:
    variant = "sports"
elif "Politics" in TAGS:
    variant = "politics"
else:
    variant = "weather"

eligible_wallets = select_strategy_eligible(wallet_stats, variant=variant)
copyable_wallets = select_strategy_selection(wallet_stats, variant=variant)
predicting_wallets = eligible_wallets - copyable_wallets

if not eligible_wallets:
    raise ValueError("No wallets passed base eligibility filters.")

_copyable_stats = wallet_stats[wallet_stats["wallet"].isin(copyable_wallets)]
_predicting_stats = wallet_stats[wallet_stats["wallet"].isin(predicting_wallets)]
print(f"variant: {variant}")
print(f"eligible wallets: {len(eligible_wallets):,}")
print(f"copyable_group: {len(copyable_wallets):,} (train copyable pnl {_copyable_stats['copyable_pnl'].sum():,.0f})")
print(f"predicting_group: {len(predicting_wallets):,} (train copyable pnl {_predicting_stats['copyable_pnl'].sum():,.0f})")


variant: sports
eligible wallets: 149
copyable_group: 92 (train copyable pnl 3,534,987)
predicting_group: 57 (train copyable pnl 1,380,262)


## Interactive wallet-filter explorer

Tune the numeric wallet filters and per-fill overlays in the browser, then click
**Save selection**. The next cell reloads the saved selection and overrides the
`FLOW_*` config defaults so the masks below follow it.


In [6]:
sys.path.insert(0, str(Path.cwd()))

from wallet_filter_explorer import open_wallet_filter_explorer

explorer = open_wallet_filter_explorer(
    wallet_stats,
    variant=variant,
    eligible_wallets=eligible_wallets,
    selected_wallets=copyable_wallets,
    fills=df,
    split_date=TRAIN_END,
    flow_price_lo=FLOW_PRICE_LO,
    flow_price_hi=FLOW_PRICE_HI,
    flow_share_thr=FLOW_SHARE_THR,
    flow_strict=FLOW_STRICT,
    out_dir=WORKSPACE_DIR,
)


Wallet filter explorer (sports) running at http://127.0.0.1:55822/
  save from the page, then re-run the load cell; stop with explorer.stop()


In [7]:
import importlib

import wallet_filter_explorer as _wfe

importlib.reload(_wfe)

selection = _wfe.load_selection(explorer.selection_path)
if selection is None:
    print(f"Nothing saved yet — use 'Save selection' in {explorer.url}")
else:
    _available = set(wallet_stats["wallet"])
    _missing = sorted(set(selection.get("passing") or []) - _available)
    if _missing:
        print(
            f"Warning: ignoring {len(_missing)} stale saved wallet(s) missing from "
            f"the current stats frame: {_missing[:5]}"
            + (" ..." if len(_missing) > 5 else "")
        )
        selection = {
            **selection,
            "passing": [w for w in selection.get("passing") or [] if w in _available],
            "picked": [w for w in selection.get("picked") or [] if w in _available],
        }

    explored_wallets = _wfe.apply_selection_filters(selection, wallet_stats, fills=df)
    print(
        f"passing={len(explored_wallets)} "
        f"picked={int(explored_wallets['picked'].sum())} "
        f"copyable_pnl={explored_wallets['copyable_pnl'].sum():,.0f}"
    )
    display(explored_wallets)


passing=49 picked=0 copyable_pnl=2,478,673


,wallet,num_buckets,trade_count,total_notional,total_pnl,copyable_pnl,num_markets,median_roi,average_roi,max_drawdown,...,copyable_pnl_similarity_loss,copyable_pnl_similarity_20m_100,copyable_pnl_similarity_profit_20m_100,copyable_pnl_similarity_loss_20m_100,max_drawdown_to_pnl,max_copyable_drawdown_to_copyable_pnl,return,buy_copyable_roi,median_dt_days_ago,picked
0,0x0d18e30e252008d83f51dae00d36873ddd9e24a6,455,3285,2509016.0153,429606.9384,143544.9203,52,0.0870,-0.0317,57992.7637,...,0.4006,NaN,NaN,NaN,0.1350,0.0415,0.1712,0.2829,110.5843,False
1,0x13e4bf768f4c2005af84d9f23c670ac0bc788b89,1822,5662,278901.0544,25567.2049,9332.9818,1069,0.1111,0.1305,7754.1655,...,0.8536,NaN,NaN,NaN,0.3033,0.6688,0.0917,0.0570,100.8430,False
2,0x1a6df901e63422055a1658b0ef2b2a871507cf4a,1472,1850,99583.7389,7474.3224,1465.4312,1051,0.0101,1.9748,95.4208,...,0.6084,NaN,NaN,NaN,0.0128,0.0188,0.0751,0.2737,252.1520,False
3,0x1c66c88a4be1ca1081d93c6e1fd5b5b5847ce371,13321,29941,687080.3073,158332.9313,94268.4535,5859,0.1236,0.0751,15154.9531,...,0.7745,NaN,NaN,NaN,0.0957,0.0850,0.2304,0.2541,137.3691,False
4,0x1f7105a18d9f36aef7c83e5df210e57cf487b2e4,1157,6634,5193534.7047,946715.6094,423944.4437,684,0.0902,0.2792,95313.4704,...,0.8056,NaN,NaN,NaN,0.1007,0.1447,0.1823,0.2043,108.4211,False
5,0x22761026b64f1f31e9d0ca7318c86922c454efd0,260,573,95881.4762,31608.7672,2979.9653,34,0.4493,0.1665,6250.8244,...,0.1798,NaN,NaN,NaN,0.1978,0.3507,0.3297,0.3462,115.8239,False
6,0x24ae647bd3d6fec6ec54e3e02606a0a260804c80,733,2559,410418.9129,107159.3798,27546.9646,469,0.1015,0.2645,10379.8544,...,0.9070,NaN,NaN,NaN,0.0969,0.4587,0.2611,0.1521,128.7232,False
7,0x25e21e9c4e469d26aaa5701948ff8923d060db79,185,194,531323.3163,31760.1140,11428.3024,128,0.1115,0.0596,4674.9390,...,0.5550,NaN,NaN,NaN,0.1472,0.2183,0.0598,0.0637,228.9073,False
8,0x2a8a536cb3448f6e0c94d4a3ed7ede7855c1cb94,919,5226,314476.9507,65270.0860,34743.3077,377,0.1111,0.1473,6026.1071,...,0.7937,NaN,NaN,NaN,0.0923,0.1176,0.2076,0.1944,125.6711,False
9,0x2d462b29127b919ae4c085e03be44ae7077e3e2d,1932,3724,238490.3109,24746.4678,-869.9174,1076,-1.0000,0.3183,3498.3454,...,0.7425,NaN,NaN,NaN,0.1414,NaN,0.1038,-0.0146,145.6816,False


## Flow-selection port (restored `stage1_flow_selection`)

Exact replica of the original flow-selection pipeline. The candidate universe is
rebuilt with the **legacy** two-frame sports rules (`legacy=True`) so this
reproduces the original ~105k test copyable PnL instead of the unified-filter
cohorts used above.

1. candidate universe via `build_sports_candidate_universe(legacy=True)`
2. uncarved BUY fills of those wallets in `primary_tag in TAGS`, lead
   `FLOWSEL_MIN_LEAD_DAYS`–`FLOWSEL_MAX_LEAD_DAYS`, `copyable_notional > 0`
3. fit / val / test split by resolution date
4. fit-profitability screen, then `size_share` threshold = fit q50
5. mask: price in `FLOWSEL_PRICE_BAND` and `size_share <= threshold`
6. wallet screen: profitable in fit and val under the mask

The final mask is joined onto the shared `df` stream by
`(wallet, condition_id, token_id, tx_hash)` and used as the `price_size`
strategy. The reference table is computed on the uncarved `fs` stream and must
reproduce `stage1_flow_selection`'s saved numbers.

In [8]:
# ── Build the flow-selection stream (exact stage1_flow_selection replica) ──
from signal_lab.candidate_universe import build_sports_candidate_universe

_fs_mdf = load_markets_processed()
_fs_mdf = _fs_mdf[_fs_mdf["primary_tag"].isin(TAGS)].copy()
flowsel_conditions = set(_fs_mdf["condition_id"])
del _fs_mdf

_, _, flowsel_candidate_wallets, _, _ = build_sports_candidate_universe(
    TRADES_DIR,
    max_lead_days=FLOWSEL_CANDIDATE_MAX_LEAD_DAYS,
    variant=variant,
    legacy=True,
)
flowsel_candidate_wallets = set(map(str, flowsel_candidate_wallets))
print(f"flow-selection candidate wallets: {len(flowsel_candidate_wallets):,}")

_fs_cols = [
    "wallet", "condition_id", "token_id", "tx_hash", "dt", "side",
    "avg_price", "trade_value_usdc", "copyable_pnl",
    "last_condition_trade_ts", "copyable_qty_5m_100", "usdc_all_60",
    "final_value_usdc", "total_quantity",
]
fs = (
    ds.dataset(TRADES_DIR, format="parquet")
    .to_table(columns=_fs_cols, filter=ds.field("wallet").isin(list(flowsel_candidate_wallets)))
    .to_pandas()
)
fs["wallet"] = fs["wallet"].astype(str)

flowsel_key_cols = ["wallet", "condition_id", "token_id", "tx_hash"]
assert not fs.duplicated(subset=flowsel_key_cols).any(), "flow-selection join key not unique"

fs = fs[fs["side"] == "BUY"].copy()
fs = fs[fs["condition_id"].isin(flowsel_conditions)].copy()
fs = fs.rename(columns={"avg_price": "price", "trade_value_usdc": "usdc_amount"})

fs["dt"] = pd.to_datetime(fs["dt"], utc=True)
fs["resolution_dt"] = pd.to_datetime(fs["last_condition_trade_ts"], utc=True)
_fs_lead = fs["resolution_dt"] - fs["dt"]
fs = fs[
    (_fs_lead <= pd.Timedelta(days=FLOWSEL_MAX_LEAD_DAYS))
    & (_fs_lead >= pd.Timedelta(days=FLOWSEL_MIN_LEAD_DAYS))
].copy()

fs["pnl"] = fs["final_value_usdc"].astype(float) - fs["usdc_amount"].astype(float)
fs["copyable_notional"] = (
    fs["price"].astype(float) * fs["copyable_qty_5m_100"].astype(float).clip(lower=0)
)
fs = fs[fs["copyable_notional"] > 0].copy()
fs["size_share"] = (
    fs["usdc_amount"].astype(float) / fs["usdc_all_60"].astype(float).replace(0, np.nan)
)

fs["is_fit"] = fs["resolution_dt"] <= FLOWSEL_FIT_END
fs["is_val"] = (fs["resolution_dt"] > FLOWSEL_FIT_END) & (fs["resolution_dt"] <= FLOWSEL_VAL_END)
fs["is_test"] = fs["resolution_dt"] > FLOWSEL_VAL_END

# Fit-profitability screen — applied BEFORE the threshold fit, exactly as in
# stage1_flow_selection.
_fs_fit_wallets = fs[fs["is_fit"]].groupby("wallet").agg(
    cpnl=("copyable_pnl", "sum"),
    fills=("copyable_pnl", "size"),
)
fs = fs[
    fs["wallet"].isin(
        _fs_fit_wallets.index[
            (_fs_fit_wallets["cpnl"] > 0)
            & (_fs_fit_wallets["fills"] >= FLOWSEL_MIN_FIT_WALLET_FILLS)
        ]
    )
].copy()
print(f"wallets after fit-profitability screen: {fs['wallet'].nunique():,}; fills: {len(fs):,}")

FLOWSEL_THR = fs[fs["is_fit"]]["size_share"].quantile(FLOWSEL_SIZE_SHARE_QUANTILE)
print(f"size_share threshold (fit q{FLOWSEL_SIZE_SHARE_QUANTILE:.0%}): {FLOWSEL_THR:.4f}")

flowsel_mask = (
    fs["price"].between(FLOWSEL_PRICE_BAND[0], FLOWSEL_PRICE_BAND[1])
    & (fs["size_share"] <= FLOWSEL_THR)
).fillna(False)

_period_cols = {"fit": "is_fit", "val": "is_val", "test": "is_test"}

def _flowsel_screen_stats(mask, period):
    return fs[mask & fs[_period_cols[period]]].groupby("wallet").agg(
        cpnl=("copyable_pnl", "sum"),
        fills=("copyable_pnl", "size"),
    )

_fs_fit = _flowsel_screen_stats(flowsel_mask, "fit")
_fs_val = _flowsel_screen_stats(flowsel_mask, "val")
_fs_fit_ok = _fs_fit.index[
    (_fs_fit["cpnl"] > 0) & (_fs_fit["fills"] >= FLOWSEL_MIN_FIT_WALLET_FILLS)
]
_fs_val_ok = _fs_val.index[
    (_fs_val["cpnl"] > 0) & (_fs_val["fills"] >= FLOWSEL_MIN_VAL_WALLET_FILLS)
]
flowsel_selected_wallets = set(_fs_fit_ok.intersection(_fs_val_ok))
flowsel_final_mask = flowsel_mask & fs["wallet"].isin(flowsel_selected_wallets)
print(f"selected wallets: {len(flowsel_selected_wallets):,}")

_reference_rows = []
for _label, _mask in [
    ("chosen filter", flowsel_mask),
    ("chosen filter + wallet screen", flowsel_final_mask),
]:
    for _period, _col in _period_cols.items():
        _sub = fs[_mask & fs[_col]]
        _notional = float(_sub["copyable_notional"].sum())
        _wallet_notional = float(_sub["usdc_amount"].sum())
        _reference_rows.append(
            {
                "variant": _label,
                "period": _period,
                "fills": len(_sub),
                "copyable_pnl": float(_sub["copyable_pnl"].sum()),
                "copyable_roi": (
                    float(_sub["copyable_pnl"].sum() / _notional)
                    if _notional > 0
                    else np.nan
                ),
                "wallet_pnl": float(_sub["pnl"].sum()),
                "wallet_roi": (
                    float(_sub["pnl"].sum() / _wallet_notional)
                    if _wallet_notional > 0
                    else np.nan
                ),
            }
        )
display(pd.DataFrame(_reference_rows).round(4))

# ── Join the final mask onto the shared df stream ──────────────────────────
_flowsel_keys = fs.loc[flowsel_final_mask, flowsel_key_cols].assign(_pass=True)
price_size_mask = (
    df.reset_index()
    .merge(_flowsel_keys, on=flowsel_key_cols, how="left")
    .set_index("index")["_pass"]
    .reindex(df.index)
    .fillna(False)
    .astype(bool)
)
price_size_wallets = set(flowsel_selected_wallets)
print(
    "price_size on shared df — "
    f"test copyable_pnl "
    f"{df.loc[~df['is_train'] & price_size_mask, 'copyable_pnl'].sum():,.0f}"
)

flow-selection candidate wallets: 35


wallets after fit-profitability screen: 31; fills: 249,791
size_share threshold (fit q50%): 0.0256
selected wallets: 15


,variant,period,fills,copyable_pnl,copyable_roi,wallet_pnl,wallet_roi
0,chosen filter,fit,33299,224518.4902,0.1150,846260.5111,0.1916
1,chosen filter,val,14250,104922.4324,0.1211,120048.5610,0.0682
2,chosen filter,test,23801,100315.3413,0.0753,428233.0266,0.1403
3,chosen filter + wallet screen,fit,27027,214111.7522,0.1220,815530.4154,0.2029
4,chosen filter + wallet screen,val,12288,100429.6698,0.1227,112947.2266,0.0667
5,chosen filter + wallet screen,test,20229,105842.9961,0.0843,426354.7741,0.1451


price_size on shared df — test copyable_pnl 105,843


/var/folders/j8/0dbnwk8n6m933m843h7hb88w0000gn/T/ipykernel_70815/1754954446.py:140: FutureWarning:

Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`



## Strategy masks

Every strategy is a mask over the shared BUY stream:

| strategy | wallets | mask |
| --- | --- | --- |
| `copy_all` | copyable_group | all BUY fills |
| `flow` | predicting_group | price band + signed flow share |
| `flow_strict` | predicting_group | flow + uptick |
| `size_share` | predicting_group | price band + small size share |
| `price_size` | flow-selection wallets | ported `stage1_flow_selection` price/size mask |
| `predicting_base` | predicting_group | all BUY fills (reference) |

Overlay thresholds are fitted on train fills only.


In [9]:
# The explorer save can override the flow config defaults.
_sel = globals().get("selection")
_sel_flow = {} if _sel is None else (_sel.get("flow") or {})
FLOW_PRICE_LO = float(_sel_flow.get("price_lo", FLOW_PRICE_LO))
FLOW_PRICE_HI = float(_sel_flow.get("price_hi", FLOW_PRICE_HI))
FLOW_SHARE_THR = float(_sel_flow.get("share_thr", FLOW_SHARE_THR))
FLOW_STRICT = bool(_sel_flow.get("strict", FLOW_STRICT))

_predicting_mask = df["wallet"].isin(predicting_wallets)

# Contrarian flow overlay.
_flow_band = df["price"].between(FLOW_PRICE_LO, FLOW_PRICE_HI)
_flow_signal = df["flow_usdc_share_60"] <= FLOW_SHARE_THR
flow_nonstrict = (_flow_band & _flow_signal).fillna(False)
flow_strict = (flow_nonstrict & (df["price_chg_last_own"] > 0)).fillna(False)

# Size-share overlay.
_size_fit = df.loc[df["is_train"] & _predicting_mask, "size_share"]
SIZE_SHARE_THR = float(_size_fit.quantile(SIZE_SHARE_QUANTILE))
size_mask = (
    df["price"].between(SIZE_PRICE_LO, SIZE_PRICE_HI)
    & (df["size_share"] <= SIZE_SHARE_THR)
).fillna(False)

_all_buy = pd.Series(True, index=df.index)
strategies = {
    "copy_all": (copyable_wallets, _all_buy),
    "predicting_base": (predicting_wallets, _all_buy),
    "flow": (predicting_wallets, flow_nonstrict),
    "flow_strict": (predicting_wallets, flow_strict),
    "size_share": (predicting_wallets, size_mask),
    "price_size": (price_size_wallets, price_size_mask),
}

print(f"flow: price {FLOW_PRICE_LO:g}-{FLOW_PRICE_HI:g}, share <= {FLOW_SHARE_THR:g}, strict={FLOW_STRICT}")
print(f"size_share: price {SIZE_PRICE_LO:g}-{SIZE_PRICE_HI:g}, thr <= {SIZE_SHARE_THR:.4f}")
print(f"price_size: flow-selection port, wallets={len(price_size_wallets):,}")
for name, (wallets, mask) in strategies.items():
    print(f"  {name:16s} wallets={len(wallets):>4,} fills={int((df['wallet'].isin(wallets) & mask).sum()):>10,}")


flow: price 0.2-0.5, share <= -0.1, strict=False
size_share: price 0.3-0.75, thr <= 0.0120
price_size: flow-selection port, wallets=15
  copy_all         wallets=  92 fills=   192,507


  predicting_base  wallets=  57 fills=   477,497
  flow             wallets=  57 fills=    90,766


  flow_strict      wallets=  57 fills=    30,690
  size_share       wallets=  57 fills=   137,336


  price_size       wallets=  15 fills=    59,544


## Unified per-strategy evaluation

Train and test rows for every strategy. `retention` is masked fills over all BUY
fills of the strategy's wallets in that period.


In [10]:
from polymarket_analysis.visualization.wallet_plots import copyable_exposure_series


def _strategy_metrics(name, wallets, mask, period):
    in_wallets = df["wallet"].isin(wallets)
    period_mask = {
        "train": df["is_train"],
        "test": ~df["is_train"],
        "all": pd.Series(True, index=df.index),
    }[period]
    base = df[in_wallets & period_mask]
    sub = df[in_wallets & period_mask & mask.fillna(False)]
    if sub.empty:
        return {
            "strategy": name, "period": period, "wallets": 0,
            "base_fills": len(base), "fills": 0, "retention": 0.0,
            "copyable_pnl": 0.0, "copyable_notional": 0.0, "roi": np.nan,
            "wallet_pnl": 0.0, "wallet_notional": 0.0, "wallet_roi": np.nan,
            "sharpe": np.nan, "max_drawdown": np.nan,
            "exposure_p50": np.nan, "exposure_p90": np.nan, "exposure_max": np.nan,
        }

    daily = sub.groupby(sub["dt"].dt.floor("D"))["copyable_pnl"].sum()
    sharpe = (
        float(daily.mean() / daily.std() * np.sqrt(365))
        if len(daily) > 1 and daily.std() > 0
        else np.nan
    )
    cumulative = daily.cumsum()
    exp_series = copyable_exposure_series(sub, "exposure")
    per_fill_exp = sub["exposure"].dropna()
    notional = (sub["price"] * sub["copyable_qty_5m_100"].clip(lower=0)).sum()
    wallet_notional = float(sub["usdc_amount"].sum())

    return {
        "strategy": name,
        "period": period,
        "wallets": int(sub["wallet"].nunique()),
        "base_fills": len(base),
        "fills": len(sub),
        "retention": len(sub) / len(base) if len(base) else np.nan,
        "copyable_pnl": float(sub["copyable_pnl"].sum()),
        "copyable_notional": float(notional),
        "roi": float(sub["copyable_pnl"].sum() / notional) if notional > 0 else np.nan,
        "wallet_pnl": float(sub["pnl"].sum()),
        "wallet_notional": wallet_notional,
        "wallet_roi": (
            float(sub["pnl"].sum() / wallet_notional) if wallet_notional > 0 else np.nan
        ),
        "sharpe": sharpe,
        "max_drawdown": float((cumulative.cummax() - cumulative).max()),
        "exposure_p50": float(per_fill_exp.quantile(0.50)) if len(per_fill_exp) else np.nan,
        "exposure_p90": float(per_fill_exp.quantile(0.90)) if len(per_fill_exp) else np.nan,
        "exposure_max": float(exp_series["exposure"].max()) if not exp_series.empty else np.nan,
    }


eval_rows = []
for _name, (_wallets, _mask) in strategies.items():
    for _period in ("train", "test"):
        eval_rows.append(_strategy_metrics(_name, _wallets, _mask, _period))

eval_df = pd.DataFrame(eval_rows)
display(eval_df.round(4))


,strategy,period,wallets,base_fills,fills,retention,copyable_pnl,copyable_notional,roi,wallet_pnl,wallet_notional,wallet_roi,sharpe,max_drawdown,exposure_p50,exposure_p90,exposure_max
0,copy_all,train,92,163164,163164,1.0000,3911823.1689,18016358.7470,0.2171,9945889.4709,41423201.1729,0.2401,6.8283,88768.4627,6.9576,200.0000,702979.0973
1,copy_all,test,46,29343,29343,1.0000,310586.6446,4065279.4317,0.0764,411326.9855,6230782.0079,0.0660,5.0708,86009.9077,10.0005,285.8829,204474.7073
2,predicting_base,train,57,337787,337787,1.0000,1911248.9777,18258124.3628,0.1047,8686203.8328,41002940.0907,0.2118,1.6572,452957.2098,6.3800,100.0000,1180716.2911
3,predicting_base,test,34,139710,139710,1.0000,-436069.2650,10468113.6078,-0.0417,292485.8442,16644051.2460,0.0176,-7.6305,482465.2325,10.0000,197.0290,478514.0874
4,flow,train,56,337787,64307,0.1904,705608.1131,2494042.5877,0.2829,2749999.3616,5864292.9584,0.4689,1.8691,121299.1103,4.1600,66.6653,267452.0150
5,flow,test,29,139710,26459,0.1894,-22762.5552,1168997.8420,-0.0195,73411.9818,1665645.6683,0.0441,-0.8870,78787.7925,7.0339,115.7780,55586.6219
6,flow_strict,train,56,337787,22301,0.0660,144519.2373,1162802.2605,0.1243,975103.4235,2342972.6468,0.4162,1.1611,58037.6640,6.6456,83.8628,127410.8562
7,flow_strict,test,27,139710,8389,0.0600,3368.0588,359713.9750,0.0094,44930.1424,511167.7472,0.0879,0.2737,28576.6177,9.0000,100.0000,17813.5633
8,size_share,train,57,337787,99212,0.2937,216371.0314,1750988.0176,0.1236,659758.5858,3110248.2374,0.2121,3.2119,12697.6183,5.0000,49.9900,107067.4027
9,size_share,test,33,139710,38124,0.2729,-3452.3667,807713.7158,-0.0043,41395.6673,1093769.2902,0.0378,-0.2668,57828.7554,8.1100,48.6082,51048.7490


## Plots

Cumulative copyable PnL and outstanding copy exposure per strategy.


In [11]:
def _strategy_fills(wallets, mask):
    return df[df["wallet"].isin(wallets) & mask.fillna(False)].copy()


fig = go.Figure()
for _name, (_wallets, _mask) in strategies.items():
    _sub = _strategy_fills(_wallets, _mask)
    if _sub.empty:
        continue
    daily = _sub.groupby(_sub["dt"].dt.floor("D"))["copyable_pnl"].sum().sort_index()
    cumulative = daily.cumsum()
    fig.add_trace(
        go.Scatter(x=cumulative.index, y=cumulative.values, mode="lines", name=f"{_name} — copyable")
    )
    daily_wallet = _sub.groupby(_sub["dt"].dt.floor("D"))["pnl"].sum().sort_index()
    wallet_cumulative = daily_wallet.cumsum()
    fig.add_trace(
        go.Scatter(
            x=wallet_cumulative.index,
            y=wallet_cumulative.values,
            mode="lines",
            name=f"{_name} — wallet",
            line=dict(dash="dot"),
        )
    )
fig.add_vline(x=TRAIN_END, line_dash="dash", line_color="black")
fig.update_layout(
    title="Cumulative PnL per strategy — copyable (solid) vs wallet (dotted)",
    xaxis_title="Date",
    yaxis_title="USDC",
    template="plotly_white",
)
fig.show()


In [12]:
fig = go.Figure()
for _name, (_wallets, _mask) in strategies.items():
    _sub = _strategy_fills(_wallets, _mask)
    _sub = _sub[~_sub["is_train"]]
    if _sub.empty:
        continue
    daily = _sub.groupby(_sub["dt"].dt.floor("D"))["copyable_pnl"].sum().sort_index()
    fig.add_trace(
        go.Scatter(
            x=daily.index,
            y=daily.cumsum().values,
            mode="lines",
            name=f"{_name} — copyable",
        )
    )
    daily_wallet = _sub.groupby(_sub["dt"].dt.floor("D"))["pnl"].sum().sort_index()
    fig.add_trace(
        go.Scatter(
            x=daily_wallet.index,
            y=daily_wallet.cumsum().values,
            mode="lines",
            name=f"{_name} — wallet",
            line=dict(dash="dot"),
        )
    )
fig.update_layout(
    title="Test period only — cumulative PnL per strategy (reset at test start)",
    xaxis_title="Date",
    yaxis_title="USDC",
    template="plotly_white",
)
fig.show()

In [13]:
fig = go.Figure()
for _name, (_wallets, _mask) in strategies.items():
    _sub = _strategy_fills(_wallets, _mask)
    if _sub.empty:
        continue
    _exp = copyable_exposure_series(_sub, "exposure", bucket_freq="D")
    if _exp.empty:
        continue
    fig.add_trace(
        go.Scatter(x=_exp["dt"], y=_exp["exposure"], mode="lines", name=_name)
    )
fig.add_vline(x=TRAIN_END, line_dash="dash", line_color="black")
fig.update_layout(
    title="Outstanding copy exposure per strategy",
    xaxis_title="Date",
    yaxis_title="USDC",
    template="plotly_white",
)
fig.show()


## price_size — test inspection (per-wallet)

Largest `price_size` wallets on the test split: per-wallet copyable vs wallet
PnL, and outstanding copy exposure over time.


In [14]:
# ── price_size on test: largest wallets — pnl, wallet pnl, exposure ─────────
from plotly.subplots import make_subplots

_ps_fills = df[df["wallet"].isin(price_size_wallets) & price_size_mask & ~df["is_train"]].copy()

_ps_rows = []
for _w, _g in _ps_fills.groupby("wallet"):
    _exp = copyable_exposure_series(_g, "exposure", bucket_freq="D")
    _notional = float((_g["price"] * _g["copyable_qty_5m_100"].clip(lower=0)).sum())
    _wallet_notional = float(_g["usdc_amount"].sum())
    _ps_rows.append(
        {
            "wallet": _w,
            "fills": len(_g),
            "copyable_pnl": float(_g["copyable_pnl"].sum()),
            "copyable_notional": _notional,
            "copyable_roi": (
                float(_g["copyable_pnl"].sum() / _notional) if _notional > 0 else np.nan
            ),
            "wallet_pnl": float(_g["pnl"].sum()),
            "wallet_notional": _wallet_notional,
            "wallet_roi": (
                float(_g["pnl"].sum() / _wallet_notional) if _wallet_notional > 0 else np.nan
            ),
            "exposure_max": float(_exp["exposure"].max()) if not _exp.empty else np.nan,
        }
    )
ps_wallet_test = (
    pd.DataFrame(_ps_rows)
    .sort_values("copyable_pnl", ascending=False)
    .reset_index(drop=True)
)
display(ps_wallet_test.round(4))

TOP_N_PS = 10
_ps_top = ps_wallet_test.head(TOP_N_PS)["wallet"].tolist()
_ps_labels = [w[:10] for w in _ps_top]
_top_share = (
    ps_wallet_test.head(TOP_N_PS)["copyable_pnl"].sum()
    / ps_wallet_test["copyable_pnl"].sum()
)
print(
    f"test copyable_pnl total {ps_wallet_test['copyable_pnl'].sum():,.0f} over "
    f"{len(ps_wallet_test)} wallets; top-{TOP_N_PS} share {_top_share:.1%}"
)

fig = make_subplots(
    rows=2,
    cols=1,
    vertical_spacing=0.12,
    subplot_titles=(
        f"Test copyable vs wallet PnL — top {len(_ps_top)} price_size wallets",
        "Outstanding copy exposure per wallet — test (daily)",
    ),
)
fig.add_trace(
    go.Bar(
        x=_ps_labels,
        y=ps_wallet_test.head(TOP_N_PS)["copyable_pnl"],
        name="copyable_pnl",
        marker_color="rgba(31,119,180,0.8)",
        offsetgroup="a",
        hovertext=_ps_top,
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Bar(
        x=_ps_labels,
        y=ps_wallet_test.head(TOP_N_PS)["wallet_pnl"],
        name="wallet_pnl",
        marker_color="rgba(255,127,14,0.6)",
        offsetgroup="b",
        hovertext=_ps_top,
    ),
    row=1,
    col=1,
)
for _w in _ps_top:
    _g = _ps_fills[_ps_fills["wallet"] == _w]
    _exp = copyable_exposure_series(_g, "exposure", bucket_freq="D")
    if _exp.empty:
        continue
    fig.add_trace(
        go.Scatter(
            x=_exp["dt"],
            y=_exp["exposure"],
            mode="lines",
            name=_w[:10],
            legendgroup=_w,
            hovertext=_w,
        ),
        row=2,
        col=1,
    )
_exp_all = copyable_exposure_series(_ps_fills, "exposure", bucket_freq="D")
if not _exp_all.empty:
    fig.add_trace(
        go.Scatter(
            x=_exp_all["dt"],
            y=_exp_all["exposure"],
            mode="lines",
            name="price_size (all)",
            line=dict(width=3, color="black"),
        ),
        row=2,
        col=1,
    )
fig.update_layout(
    barmode="group",
    title="price_size — test inspection (largest wallets)",
    yaxis_title="USDC (PnL)",
    yaxis2_title="USDC (exposure)",
    template="plotly_white",
    height=750,
)
fig.show()


,wallet,fills,copyable_pnl,copyable_notional,copyable_roi,wallet_pnl,wallet_notional,wallet_roi,exposure_max
0,0x076daa87c4fe1a85402a9b6b8e0a866224388d4c,11703,63177.2287,728306.8547,0.0867,351603.9330,2270502.6081,0.1549,32174.2411
1,0xf3ce7f04dde4f8c5aed19a20e5ecd8520e5ca57a,108,22996.9406,55109.4960,0.4173,23964.9448,58568.0547,0.4092,13833.1311
2,0xc34b4901be324718c176100e350211d97b199296,998,15414.9278,52574.5926,0.2932,22928.9285,136078.7640,0.1685,9427.7357
3,0xf49614e63fb15383d4a9b717a1be03ad2410fe79,4058,9595.4907,232144.0030,0.0413,20909.0643,258026.6656,0.0810,19973.9500
4,0x1f7105a18d9f36aef7c83e5df210e57cf487b2e4,116,9471.1479,10539.0189,0.8987,10660.4530,11396.3746,0.9354,6301.0083
5,0x73492e9ea5e7423cc0bb10a7169a0c81972dcc3a,269,3557.0093,14457.1261,0.2460,4463.3380,15461.0523,0.2887,2952.2151
6,0x2a8a536cb3448f6e0c94d4a3ed7ede7855c1cb94,133,2536.5184,5926.7842,0.4280,2401.4368,6588.4797,0.3645,2348.5774
7,0x58194e6e2a57f3f30ad0ac379129c20c03832b4f,990,1852.0211,18156.0541,0.1020,2021.7876,19058.1952,0.1061,1402.3288
8,0x7cb6b8ed713dbb9bb1315b86ed05b666fafc96d1,55,881.3612,4788.6140,0.1841,1061.9142,6025.7085,0.1762,503.5593
9,0x24ae647bd3d6fec6ec54e3e02606a0a260804c80,299,422.9181,11936.0647,0.0354,1234.7572,15118.1535,0.0817,4618.5561


test copyable_pnl total 105,843 over 15 wallets; top-10 share 122.7%


## Export price_size wallets (scaled buy-copy JSON)

Writes the `price_size` cohort in the `stage1_scaled_result_sport.json` shape
loaded by PolyBot's `ScaledBuyCopyConfig` (`Stage1ScaledResultLoader`):
per-wallet `alpha_kelly`/`alpha_tier` (same math as `signal_lab.wallet_scaling`,
fit window = flow-selection fit, daily PnL by release date), `metadata.tags`
for the strategy tag gate, and a `mask` section (price band + size-share
threshold) that gates the bot trigger exactly like the notebook strategy.


In [15]:
import json
from datetime import datetime, timezone

# ── Per-wallet stats on the price_size masked stream ────────────────────────
ps = df[df["wallet"].isin(price_size_wallets) & price_size_mask].copy()
ps["rel_date"] = ps["last_condition_trade_ts"].dt.floor("D")

ps_fit = ps[ps["last_condition_trade_ts"] <= FLOWSEL_FIT_END]
ps_test = ps[ps["last_condition_trade_ts"] > FLOWSEL_VAL_END]

# Fit-window daily pnl by release date — same basis as wallet_scaling.
fit_daily = ps_fit.groupby(["wallet", "rel_date"])["copyable_pnl"].sum().reset_index()
st = fit_daily.groupby("wallet")["copyable_pnl"].agg(
    mu="mean", sigma="std", n_days="size", total_pnl="sum",
)
st["sigma"] = st["sigma"].fillna(0.0)
_n_w = st["n_days"].sum()
_mu_cs = float((st["mu"] * st["n_days"]).sum() / _n_w) if _n_w > 0 else 0.0
_var_cs = float((st["sigma"] ** 2 * st["n_days"]).sum() / _n_w) if _n_w > 0 else 1.0
_LAMBDA_MU, _LAMBDA_VAR = 20.0, 20.0
st["mu_shrunk"] = (st["n_days"] * st["mu"] + _LAMBDA_MU * _mu_cs) / (st["n_days"] + _LAMBDA_MU)
st["var_shrunk"] = (st["n_days"] * st["sigma"] ** 2 + _LAMBDA_VAR * _var_cs) / (
    st["n_days"] + _LAMBDA_VAR
)
st["sharpe_proxy"] = st["mu_shrunk"] / np.sqrt(st["var_shrunk"])


def _normalize_mean1(alpha):
    m = alpha.mean()
    return alpha / m if m > 0 else alpha


ALPHA_MAX = 1.0
st["alpha_kelly"] = _normalize_mean1(
    np.clip(st["mu_shrunk"].clip(lower=0.0) / st["var_shrunk"], 0.0, ALPHA_MAX)
)

# tier3@1-0.25 (stage1_scaled TIER_GRID mirror)
_N_TIERS, _TIER_AMAX, _TIER_AMIN = 3, 1.0, 0.25
_ranked = st["sharpe_proxy"].rank(method="first", ascending=False).astype(int) - 1
_size = max(int(np.ceil(len(st) / _N_TIERS)), 1)
_tier = np.minimum(_ranked // _size, _N_TIERS - 1)
_vals = _TIER_AMIN + (_TIER_AMAX - _TIER_AMIN) * (_N_TIERS - 1 - np.arange(_N_TIERS)) / (_N_TIERS - 1)
st["alpha_tier"] = _normalize_mean1(pd.Series(_vals[_tier.to_numpy()], index=st.index))

# Test-window stats (release-date daily basis)
test_daily = ps_test.groupby(["wallet", "rel_date"])["copyable_pnl"].sum().reset_index()
test_st = test_daily.groupby("wallet")["copyable_pnl"].agg(test_pnl="sum", test_n_days="size")
test_sharpe = (
    test_daily.groupby("wallet")["copyable_pnl"]
    .apply(
        lambda s: (s.mean() / s.std() * np.sqrt(365.0))
        if s.std() > 0 and len(s) >= 2
        else np.nan
    )
    .rename("test_sharpe")
)
contrib = st.join(test_st, how="left").join(test_sharpe, how="left")
contrib["test_pnl"] = contrib["test_pnl"].fillna(0.0)
contrib["test_n_days"] = contrib["test_n_days"].fillna(0)
contrib = contrib.reset_index()
contrib["test_roi"] = contrib["test_pnl"] / contrib["total_pnl"].replace(0, np.nan)

# ── Payload (mirrors stage1_scaled_result_sport.json + mask section) ────────
ps_test_row = eval_df[(eval_df["strategy"] == "price_size") & (eval_df["period"] == "test")].iloc[0]

test_performance = {
    "config": "price_size (mask, alpha_kelly@1)",
    "trades": int(ps_test_row["fills"]),
    "pnl": float(ps_test_row["copyable_pnl"]),
    "roi_w": float(ps_test_row["roi"]),
    "sharpe_daily": float(ps_test_row["sharpe"]),
    "wallet_pnl": float(ps_test_row["wallet_pnl"]),
    "retention": float(ps_test_row["retention"]),
}

best_params = {
    "price_band": [float(FLOWSEL_PRICE_BAND[0]), float(FLOWSEL_PRICE_BAND[1])],
    "size_share_threshold": float(FLOWSEL_THR),
    "size_share_quantile": float(FLOWSEL_SIZE_SHARE_QUANTILE),
    "min_fit_wallet_fills": int(FLOWSEL_MIN_FIT_WALLET_FILLS),
    "min_val_wallet_fills": int(FLOWSEL_MIN_VAL_WALLET_FILLS),
}

# Consumed by Stage1ScaledResultLoader -> ScaledBuyCopyTriggerStrategy mask gate.
mask_section = {
    "price_lo": float(FLOWSEL_PRICE_BAND[0]),
    "price_hi": float(FLOWSEL_PRICE_BAND[1]),
    "size_share_max": float(FLOWSEL_THR),
    "size_share_window_s": 60,
}

metadata = {
    "type": "scaled_copy",
    "strategy": "price_size",
    "market_filter_mode": MARKET_FILTER_MODE.value,
    "saved_strategy": "price_size",
    "tags": sorted(TAGS),
    "quote_ttl": 300,
    "run_timestamp": datetime.now(timezone.utc).isoformat(),
    "n_wallets_selected": int((contrib["alpha_kelly"] > 0).sum()),
    "n_wallets_total": len(price_size_wallets),
    "split_sizes": {
        "fit": int((fs["is_fit"] & flowsel_final_mask).sum()),
        "val": int((fs["is_val"] & flowsel_final_mask).sum()),
        "test": int((fs["is_test"] & flowsel_final_mask).sum()),
    },
}


def _convert(obj):
    if isinstance(obj, (float, np.floating)) and np.isnan(obj):
        return None
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, (np.bool_,)):
        return bool(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    return obj


wallet_cols = [
    "wallet", "mu", "sigma", "n_days", "total_pnl", "sharpe_proxy",
    "alpha_kelly", "alpha_tier", "test_pnl", "test_n_days", "test_sharpe", "test_roi",
]
wallet_records = contrib[[c for c in wallet_cols if c in contrib.columns]].to_dict(orient="records")
wallet_records = [{k: _convert(v) for k, v in w.items()} for w in wallet_records]

payload = {
    "stage": 1,
    "best_params": {k: _convert(v) for k, v in best_params.items()},
    # No val sizing calibration: wallets are screened on fit+val, alpha grid is fixed.
    "best_scale_sharpe": None,
    "mask": {k: _convert(v) for k, v in mask_section.items()},
    "test_performance": {k: _convert(v) for k, v in test_performance.items()},
    "metadata": metadata,
    "wallets": wallet_records,
}

_MODE_RESULT_SUFFIX = {
    MarketFilterMode.DEFAULT: "",
    MarketFilterMode.SPORTS: "_sport",
    MarketFilterMode.CRYPTO: "_crypto",
}
out_path = Path.cwd().parent / "wallet_selection" / f"stage1_price_size{_MODE_RESULT_SUFFIX[MARKET_FILTER_MODE]}.json"
out_path.parent.mkdir(parents=True, exist_ok=True)
with open(out_path, "w") as f:
    json.dump(payload, f, indent=2)
print(f"Saved price_size scaled-buy-copy result -> {out_path.resolve()}")
print(f"wallets: {len(wallet_records)} (alpha_kelly>0: {metadata['n_wallets_selected']})")
display(contrib[["wallet", "alpha_kelly", "alpha_tier", "total_pnl", "test_pnl", "test_sharpe"]].round(4))


Saved price_size scaled-buy-copy result -> /Users/vobornij/projects/polymarket/notebooks/wallet_selection/stage1_price_size_sport.json
wallets: 15 (alpha_kelly>0: 15)


,wallet,alpha_kelly,alpha_tier,total_pnl,test_pnl,test_sharpe
0,0x076daa87c4fe1a85402a9b6b8e0a866224388d4c,0.2439,1.6000,62091.5291,63177.2287,4.4277
1,0x149a9ec31ae097faa33b2f2ae8df659889c149d0,0.6982,0.4000,1749.4897,-1987.7616,-1.5693
2,0x1f7105a18d9f36aef7c83e5df210e57cf487b2e4,0.7607,1.6000,37275.6825,9471.1479,12.4048
3,0x24ae647bd3d6fec6ec54e3e02606a0a260804c80,0.6993,1.0000,7215.0644,422.9181,0.4958
4,0x2a8a536cb3448f6e0c94d4a3ed7ede7855c1cb94,2.1995,1.6000,17833.2928,2536.5184,6.2539
5,0x3d0a2cfe37769d915b40c49fe85b3602d53e1bd3,0.6302,0.4000,450.1037,-340.1460,-1.8105
6,0x46ca395d519ecc8fa9a4fa5c77917f762cf8ac4c,0.5809,1.0000,633.9988,-1603.1012,-3.9256
7,0x58194e6e2a57f3f30ad0ac379129c20c03832b4f,0.5907,0.4000,121.9679,1852.0211,3.1962
8,0x58ce8cec3cbc6eb3dbbe8cd39436bc0dd2ebe206,1.0002,1.0000,4441.6858,-6662.1042,-6.6140
9,0x73492e9ea5e7423cc0bb10a7169a0c81972dcc3a,2.5112,1.6000,39577.1130,3557.0093,6.0779


## Save stage2 cohorts

`stage2_backtest_simplified.ipynb` consumes `selected_wallets_v2.parquet`
(`wallet`, `wallet_group`).


In [16]:
copyable_df = (
    wallet_stats[wallet_stats["wallet"].isin(copyable_wallets)]
    .copy()
    .assign(wallet_group="copyable_group")
)
predicting_df = (
    wallet_stats[wallet_stats["wallet"].isin(predicting_wallets)]
    .copy()
    .assign(wallet_group="predicting_group")
)

selected_wallets = (
    pd.concat([copyable_df, predicting_df], ignore_index=True)
    .drop_duplicates(subset=["wallet", "wallet_group"])
    .reset_index(drop=True)
)

_out = WORKSPACE_DIR / "selected_wallets_v2.parquet"
selected_wallets.to_parquet(_out, index=False)
print(f"Saved {_out}: {len(selected_wallets):,} wallets")
print(selected_wallets["wallet_group"].value_counts().to_dict())
selected_wallets[["wallet", "wallet_group"]].head()


Saved ../../data/trade_signals_workspace_v2_sports/selected_wallets_v2.parquet: 149 wallets
{'copyable_group': 92, 'predicting_group': 57}


,wallet,wallet_group
0,0x3ef7342f161113628ef96129f43fded49acdcde6,copyable_group
1,0x83393548800a7f1cda599a27012a7aa555e7caa9,copyable_group
2,0x343d4466dc323b850e5249394894c7381d91456e,copyable_group
3,0x3d4c6daaef6feac0f60ea232ed478eac682041c4,copyable_group
4,0x2f7709800bdd3cf81b1c89c4f8b94b36caf59da4,copyable_group
